# Notebook 19 — Probability calibration and leakage-safe temporal stacking

**Development-only.** Nested inner chronological preprocessing, separate stacking and calibration windows. Final historical holdout and prospective ledger are not accessed.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=="notebooks": ROOT=ROOT.parent
if not (ROOT/"src").exists(): raise FileNotFoundError("Open notebook from repository root or notebooks/ directory")
sys.path.insert(0,str(ROOT/"src"))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from stock_ml_dynamic.temporal_calibration import run_temporal_calibration
from stock_ml_dynamic.pooled_models import load_master
print("Project root:",ROOT)

## Experiment settings

Start with the smoke test; switch to the full experiment after confirming correctness.

In [ ]:
SMOKE_TEST=True
N_FOLDS=2 if SMOKE_TEST else 5
MAX_TRAIN_ROWS=25000 if SMOKE_TEST else 60000
TREE_TRAIN_ROWS=15000 if SMOKE_TEST else 35000
SEED=42
OUTPUT=ROOT/"artifacts"/"temporal_calibration"/"1d"
required=("master_labeled.parquet","latest_inference.parquet","transformers.joblib","manifest.json")
candidates=[ROOT/"artifacts"/"master_tables"/"1d",ROOT/"notebooks"/"artifacts"/"master_tables"/"1d"]
master_dir=next((p for p in candidates if all((p/f).exists() for f in required)),None)
if master_dir is None:raise FileNotFoundError("Notebook 03 master table artifacts are required")
master,latest,bundle,source_manifest=load_master(master_dir)
print("Rows:",len(master),"symbols:",master.symbol.nunique())

## Run strictly chronological nested calibration and stacking

In [ ]:
scores,summary,manifest,boundary=run_temporal_calibration(master,n_folds=N_FOLDS,max_train_rows=MAX_TRAIN_ROWS,tree_train_rows=TREE_TRAIN_ROWS,seed=SEED,output_dir=OUTPUT)
print("Development boundary:",boundary)
print("Final holdout and prospective ledger untouched")

## Compare discrimination, calibration, and accuracy

In [ ]:
display(summary)
display(manifest)
fig,ax=plt.subplots(figsize=(10,4))
ax.bar(summary.model,summary.roc_auc_mean)
ax.set_ylabel("Mean macro OVR ROC-AUC")
ax.tick_params(axis="x",rotation=35)
plt.tight_layout();plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(10,4))
ax.bar(summary.model,summary.log_loss_mean)
ax.set_ylabel("Mean multiclass log loss (lower better)")
ax.tick_params(axis="x",rotation=35)
plt.tight_layout();plt.show()

## Interpretation

Calibration should be assessed primarily through log loss and ECE, while ROC-AUC measures discrimination. Do not claim a winner based on tiny mean differences. All outcomes here are historical development folds; deployment and the prospective ledger remain frozen.